<a href="https://colab.research.google.com/github/douglaskorvo/tourism_supply_chain/blob/main/00_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data collection — Google Places review corpus (BRICS+ tourism providers)

**Stage 1 of 2.** This notebook documents and reproduces the retrieval procedure that produced `reviews_raw.csv`, the input to the analysis notebook (`tssc_analysis.ipynb`).

Companion to the manuscript submitted to *Sustainability* (MDPI), Special Issue *Sustainable Management of Logistics and Supply Chain*.

Douglas S. Rodrigues — Production Engineering Department, Fluminense Federal University

---

## Read this before running

**This notebook is documentation first and executable second.** It runs in `DRY_RUN` mode by default: it prints the exact requests that would be issued, without contacting the API and without incurring cost. Set `DRY_RUN = False` and provide an API key only if you intend to collect a new corpus.

**Re-running will not reproduce the published corpus.** Google Places returns a relevance-ranked subset of reviews that changes continuously as users post, edit and delete content, and as the platform re-ranks. A new run yields a *new sample* from the same population and protocol, not a copy of the original. This is a property of the source, not a defect of the code, and it is the reason the collected corpus is archived rather than regenerated.

**Cost.** A full run issues 899 city–query combinations (29 cities × 31 queries), each paginated up to three pages, for up to ~2,700 Text Search requests, plus one Place Details request per provider–stratum pair (~12,150 in the published corpus). Check current Google Maps Platform pricing before running: at the rates in force when this corpus was collected, a full run was in the order of hundreds of US dollars.

**Terms of service.** Google Maps Platform terms restrict caching and redistribution of Places content. This is why the raw corpus is not deposited in the public repository and is instead made available to reviewers and editors on request.

## 1. Environment and credentials

In [ ]:
import os, re, json, time, math, hashlib, random
from typing import List, Dict, Any, Tuple
from datetime import datetime, timezone
from pathlib import Path

import requests
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

# ---------------------------------------------------------------- run settings
DRY_RUN = True          # True = print the requests, contact nothing, spend nothing
OUTDIR  = Path("output"); OUTDIR.mkdir(exist_ok=True)

# The key is read from the environment. Never hard-code it in a notebook that
# will be committed to a repository.
#   Linux / macOS : export GOOGLE_PLACES_KEY="..."
#   Windows       : setx GOOGLE_PLACES_KEY "..."
#   Colab         : from google.colab import userdata; os.environ["GOOGLE_PLACES_KEY"] = userdata.get("GOOGLE_PLACES_KEY")
GOOGLE_KEY = os.getenv("GOOGLE_PLACES_KEY", "")

print(f"DRY_RUN = {DRY_RUN}")
print(f"API key present: {bool(GOOGLE_KEY)}")
if not DRY_RUN and not GOOGLE_KEY:
    raise RuntimeError("Set GOOGLE_PLACES_KEY before running with DRY_RUN = False.")

### A note on API versions

The corpus was collected with the **Places API (Legacy)** web-service endpoints, `place/textsearch/json` and `place/details/json`.

Google froze the legacy Places API in March 2025: it continues to operate for Cloud projects that already had it enabled, but it **cannot be enabled in new Cloud projects**. Anyone replicating this protocol from scratch will therefore need **Places API (New)** (`places.googleapis.com/v1/places:searchText` and `v1/places/{place_id}`), whose request and response schemas differ. Section 7 maps the fields used here onto their counterparts in the new API.

The legacy endpoints are retained in this notebook because they are what actually produced the corpus. Documenting the procedure as executed takes precedence over documenting the procedure as it would be written today.

## 2. Sampling frame

Twenty-nine cities across ten emerging economies. Each entry gives the centroid latitude and longitude and the search radius in metres. Radii were set to 25 km for large metropolitan areas, 20 km for medium cities and 15 km for Lalibela, whose tourism cluster is compact.

These coordinates and radii are the geographic component of the sampling frame and should be reported in the manuscript appendix.

In [ ]:
COUNTRIES: Dict[str, Dict[str, Tuple[float, float, int]]] = {
    "Brazil": {
        "Rio de Janeiro":   (-22.9068, -43.1729, 25000),
        "São Paulo":        (-23.5505, -46.6333, 25000),
        "Salvador":         (-12.9777, -38.5016, 20000),
        "Foz do Iguaçu":    (-25.5163, -54.5854, 20000),
    },
    "Russia": {
        "Moscow":           (55.7558, 37.6173, 25000),
        "Saint Petersburg": (59.9311, 30.3609, 25000),
    },
    "India": {
        "Delhi":            (28.6139, 77.2090, 25000),
        "Agra":             (27.1767, 78.0081, 20000),
        "Jaipur":           (26.9124, 75.7873, 20000),
        "Mumbai":           (19.0760, 72.8777, 25000),
    },
    "China": {
        "Beijing":          (39.9042, 116.4074, 25000),
        "Xi'an":            (34.3416, 108.9398, 20000),
        "Shanghai":         (31.2304, 121.4737, 25000),
    },
    "South Africa": {
        "Cape Town":        (-33.9249, 18.4241, 25000),
        "Johannesburg":     (-26.2041, 28.0473, 25000),
    },
    "Egypt": {
        "Cairo":            (30.0444, 31.2357, 25000),
        "Luxor":            (25.6872, 32.6396, 20000),
        "Aswan":            (24.0889, 32.8998, 20000),
    },
    "Ethiopia": {
        "Addis Ababa":      (8.9806, 38.7578, 25000),
        "Lalibela":         (12.0317, 39.0476, 15000),
    },
    "Saudi Arabia": {
        "Riyadh":           (24.7136, 46.6753, 25000),
        "Jeddah":           (21.4858, 39.1925, 25000),
        "Al-Ula":           (26.6167, 37.9167, 20000),
    },
    "United Arab Emirates": {
        "Dubai":            (25.2048, 55.2708, 25000),
        "Abu Dhabi":        (24.4539, 54.3773, 25000),
        "Sharjah":          (25.3463, 55.4209, 20000),
    },
    "Iran": {
        "Tehran":           (35.6892, 51.3890, 25000),
        "Isfahan":          (32.6539, 51.6660, 20000),
        "Shiraz":           (29.5918, 52.5837, 20000),
    },
}

n_cities = sum(len(v) for v in COUNTRIES.values())
print(f"{len(COUNTRIES)} countries, {n_cities} cities")

## 3. Search strata

The two collection strata are defined operationally by these query lists. **This is the actual operationalisation of the creative-experience and mass-market configurations** — the labels used throughout the analysis mean "retrieved by this list of queries", nothing more. The lists therefore belong in the manuscript appendix, not only in the code.

Queries are bilingual (English and Portuguese) to improve recall in Brazilian cities, where Portuguese-language business names and categories predominate.

Note an asymmetry worth reporting: the creative list has 18 queries and the mass-market list 13. Query counts are not balanced across strata, so the number of providers retrieved per stratum reflects both the underlying supply of providers and the breadth of the query list.

In [ ]:
QUERIES: Dict[str, List[str]] = {
    "mass_market": [
        "city tour", "hop-on hop-off", "bus tour", "big bus", "tour operator",
        "sightseeing tour", "guided tour", "group tour",
        "ônibus turístico", "turismo em massa", "passeio em grupo",
        "excursion", "package tour",
    ],
    "creative": [
        "cultural workshop", "craft workshop", "artisan workshop", "community-based tourism",
        "cooking class", "dance class", "handicraft", "weaving", "pottery",
        "food experience", "homestay experience",
        "oficina cultural", "artesanato", "oficina de artesanato", "cozinha local",
        "aula de dança", "experiência gastronômica", "turismo de base comunitária",
    ],
}

for seg, q in QUERIES.items():
    print(f"{seg:12s} {len(q):2d} queries")
print(f"\nplanned Text Search calls: {n_cities} cities x {sum(len(q) for q in QUERIES.values())} queries "
      f"x up to 3 pages = up to {n_cities * sum(len(q) for q in QUERIES.values()) * 3:,}")

## 4. Retrieval parameters

Three parameters determine what the corpus contains, and each has a consequence that must be reported in the manuscript.

| Parameter | Value | Consequence |
|---|---|---|
| `TEXTSEARCH_LANG` | `en` | Provider discovery is language-neutral, avoiding bias toward providers with English-language listings in non-anglophone markets |
| `DETAILS_LANG` | `pt-BR` | **Google returns review text machine-translated into Brazilian Portuguese.** This is the origin of the translation variable in the analysis: it is a deliberate request parameter, not an artefact |
| `MAX_PAGES` | 3 | Text Search returns 20 results per page, so at most 60 providers per query per city |

A fourth constraint is imposed by the platform: **Place Details returns at most five reviews per provider**, ranked by platform-defined relevance. The corpus is therefore a relevance-ranked subset, not a random sample of each provider's reviews.

In [ ]:
TEXTSEARCH_LANG = "en"      # provider discovery
DETAILS_LANG    = "pt-BR"   # review retrieval -> triggers machine translation
MAX_PAGES       = 3         # 20 results per page
PAGE_TOKEN_WAIT = 2.0       # seconds; Google requires a delay before a next_page_token is valid
REQUEST_PAUSE   = 0.05      # polite pacing between calls
MAX_RETRIES     = 4         # exponential backoff on transient failures

TEXTSEARCH_URL = "https://maps.googleapis.com/maps/api/place/textsearch/json"
DETAILS_URL    = "https://maps.googleapis.com/maps/api/place/details/json"

DETAILS_FIELDS = ("name,types,formatted_address,geometry,"
                  "international_phone_number,website,rating,user_ratings_total,reviews")

## 5. Retrieval functions

The two functions below are faithful to the procedure that produced the corpus. Three elements were **added** for robustness and provenance and are marked in the code; they change reliability, not sampling behaviour:

- retry with exponential backoff on transient errors and `OVER_QUERY_LIMIT`;
- a call counter and an error log, so a run can be audited afterwards;
- `DRY_RUN` short-circuiting.

In [ ]:
CALLS = {"textsearch": 0, "details": 0, "errors": 0}
ERROR_LOG: List[Dict[str, Any]] = []

def _get(url: str, params: Dict[str, Any], kind: str) -> Dict[str, Any]:
    """GET with exponential backoff. [added for robustness]"""
    if DRY_RUN:
        shown = {k: ("<KEY>" if k == "key" else v) for k, v in params.items()}
        print(f"  [dry-run] GET {url}\n            {shown}")
        return {"status": "DRY_RUN", "results": [], "result": {}}

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=30)
            data = r.json()
            CALLS[kind] += 1
            status = data.get("status")
            if status in ("OK", "ZERO_RESULTS"):
                time.sleep(REQUEST_PAUSE)
                return data
            if status == "OVER_QUERY_LIMIT":
                wait = (2 ** attempt) + random.random()
                print(f"  OVER_QUERY_LIMIT — waiting {wait:.1f}s")
                time.sleep(wait); continue
            CALLS["errors"] += 1
            ERROR_LOG.append({"url": url, "status": status,
                              "message": data.get("error_message"), "params": str(params)[:200]})
            return data
        except requests.RequestException as e:
            wait = (2 ** attempt) + random.random()
            print(f"  network error ({e}) — retrying in {wait:.1f}s")
            time.sleep(wait)
    CALLS["errors"] += 1
    ERROR_LOG.append({"url": url, "status": "MAX_RETRIES", "params": str(params)[:200]})
    return {"status": "MAX_RETRIES", "results": [], "result": {}}


def textsearch_paginated(query: str, lat: float, lng: float, radius_m: int,
                         key: str, max_pages: int = MAX_PAGES,
                         lang: str = TEXTSEARCH_LANG) -> List[Dict[str, Any]]:
    """Up to 20 results per page, up to `max_pages` pages."""
    params = {"query": query, "location": f"{lat},{lng}", "radius": radius_m,
              "key": key, "language": lang}
    out, pages = [], 0
    while True:
        data = _get(TEXTSEARCH_URL, params, "textsearch")
        out.extend(data.get("results", []))
        token = data.get("next_page_token")
        pages += 1
        if not token or pages >= max_pages:
            break
        time.sleep(PAGE_TOKEN_WAIT)          # required before the token becomes valid
        params = {"pagetoken": token, "key": key}
    return out


def get_place_details(place_id: str, key: str, language: str = DETAILS_LANG) -> Dict[str, Any]:
    """Returns provider attributes and up to five relevance-ranked reviews."""
    params = {"place_id": place_id, "fields": DETAILS_FIELDS, "language": language, "key": key}
    data = _get(DETAILS_URL, params, "details")
    if data.get("status") not in ("OK", "DRY_RUN"):
        return {"place_id": place_id, "error": f"{data.get('status')} - {data.get('error_message')}"}
    return data.get("result", {})

print("retrieval functions defined")

## 6. Collection pipeline

Two passes. The first discovers providers by city and stratum; the second retrieves provider details and reviews.

Deduplication at this stage is on the pair (`place_id`, `segment`) — **not** on `place_id` alone. A provider matched by queries from both strata is therefore detailed twice and contributes up to ten reviews rather than five. Nine providers behaved this way in the published corpus; they are identified and handled in the analysis notebook.

In [ ]:
def discover_places() -> pd.DataFrame:
    """Pass 1 — Text Search by city and stratum."""
    rows: List[Dict[str, Any]] = []
    combos = [(c, city, lat, lng, rad, seg, q)
              for c, cities in COUNTRIES.items()
              for city, (lat, lng, rad) in cities.items()
              for seg, qs in QUERIES.items()
              for q in qs]

    for country, city, lat, lng, rad, seg, q in tqdm(combos, desc="Text Search"):
        for r in textsearch_paginated(q, lat, lng, rad, GOOGLE_KEY):
            pid = r.get("place_id")
            if not pid:
                continue
            rows.append({
                "country": country, "city": city, "segment": seg, "query": q,
                "place_id": pid,
                "name_hint": r.get("name"),
                "rating_hint": r.get("rating"),
                "user_ratings_total_hint": r.get("user_ratings_total"),
                "types_hint": ",".join(r.get("types", []) or []),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    # one row per provider per stratum; a provider found by several queries is kept once
    return df.drop_duplicates(subset=["place_id", "segment"]).reset_index(drop=True)


def fetch_details_and_reviews(df_places: pd.DataFrame):
    """Pass 2 — Place Details for every (provider, stratum) pair."""
    rows_places, rows_reviews = [], []

    for _, row in tqdm(df_places.iterrows(), total=len(df_places), desc="Place Details"):
        det = get_place_details(row.place_id, GOOGLE_KEY)
        if "error" in det:
            rows_places.append({**row.to_dict(), "error": det["error"]})
            continue

        loc = (det.get("geometry", {}) or {}).get("location", {}) or {}
        rows_places.append({
            **row.to_dict(),
            "name": det.get("name"),
            "formatted_address": det.get("formatted_address"),
            "lat": loc.get("lat"), "lng": loc.get("lng"),
            "rating": det.get("rating"),
            "user_ratings_total": det.get("user_ratings_total"),
            "website": det.get("website"),
            "phone": det.get("international_phone_number"),
            "types": ",".join(det.get("types", []) or []),
        })

        for rv in det.get("reviews", []) or []:
            rows_reviews.append({
                "place_id":    row.place_id,
                "country":     row.country,
                "city":        row.city,
                "segment":     row.segment,
                "author_name": rv.get("author_name"),
                "rating":      rv.get("rating"),
                "text":        rv.get("text"),
                "time_desc":   rv.get("relative_time_description"),
                "lang":        rv.get("language"),
            })

    return pd.DataFrame(rows_places), pd.DataFrame(rows_reviews)

print("pipeline functions defined")

In [ ]:
# ------------------------------------------------------------------ execute
t0 = datetime.now(timezone.utc)

if DRY_RUN:
    print("DRY RUN — showing the first three requests of pass 1, then stopping.\n")
    demo = [("Brazil", "Rio de Janeiro", -22.9068, -43.1729, 25000, "creative", q)
            for q in QUERIES["creative"][:3]]
    for country, city, lat, lng, rad, seg, q in demo:
        print(f"{country} / {city} / {seg} / '{q}'")
        textsearch_paginated(q, lat, lng, rad, "<KEY>", max_pages=1)
    print("\nThen, for each provider found:")
    get_place_details("<PLACE_ID>", "<KEY>")
    df_places = df_details = df_reviews = pd.DataFrame()
else:
    df_places  = discover_places()
    print(f"\nunique (provider, stratum) pairs: {len(df_places):,} "
          f"| unique providers: {df_places.place_id.nunique():,}")
    df_details, df_reviews = fetch_details_and_reviews(df_places)

t1 = datetime.now(timezone.utc)
print(f"\nelapsed: {(t1 - t0).total_seconds():.0f}s | API calls: {CALLS}")

### Output files

`reviews_raw.csv` carries the nine columns consumed by the analysis notebook. `places_details.csv` is retained for provenance and for the provider-level validation of the stratum labels described in the manuscript.

In [ ]:
REVIEW_COLS = ["place_id", "country", "city", "segment",
               "author_name", "rating", "text", "time_desc", "lang"]

if not DRY_RUN and not df_reviews.empty:
    df_reviews[REVIEW_COLS].to_csv(OUTDIR / "reviews_raw.csv", index=False, encoding="utf-8")
    df_details.to_csv(OUTDIR / "places_details.csv", index=False, encoding="utf-8")

    # provenance manifest — record what was run, when, and against which configuration
    cfg_blob = json.dumps({"countries": COUNTRIES, "queries": QUERIES,
                           "textsearch_lang": TEXTSEARCH_LANG, "details_lang": DETAILS_LANG,
                           "max_pages": MAX_PAGES}, sort_keys=True, ensure_ascii=False)
    manifest = {
        "collected_utc": t0.isoformat(),
        "duration_seconds": round((t1 - t0).total_seconds()),
        "api": "Google Places API (Legacy) — textsearch/json + details/json",
        "config_sha256": hashlib.sha256(cfg_blob.encode()).hexdigest(),
        "countries": len(COUNTRIES), "cities": n_cities,
        "queries_mass_market": len(QUERIES["mass_market"]),
        "queries_creative": len(QUERIES["creative"]),
        "provider_stratum_pairs": int(len(df_places)),
        "unique_providers": int(df_places.place_id.nunique()),
        "reviews": int(len(df_reviews)),
        "api_calls": CALLS,
        "errors": len(ERROR_LOG),
    }
    (OUTDIR / "collection_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    if ERROR_LOG:
        pd.DataFrame(ERROR_LOG).to_csv(OUTDIR / "collection_errors.csv", index=False)

    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print("dry run — nothing written. Set DRY_RUN = False to collect.")

## 7. Migrating to Places API (New)

Because the legacy endpoints cannot be enabled in new Cloud projects, a fresh replication requires the new API. The field mapping is direct for everything used here.

| Legacy | Places API (New) |
|---|---|
| `GET place/textsearch/json?query=&location=&radius=` | `POST v1/places:searchText` with `textQuery` and a `locationBias.circle` |
| `GET place/details/json?place_id=&fields=` | `GET v1/places/{place_id}` with a `X-Goog-FieldMask` header |
| `language` | `languageCode` |
| `next_page_token` | `nextPageToken` |
| `result.reviews[].author_name` | `reviews[].authorAttribution.displayName` |
| `result.reviews[].text` | `reviews[].text.text` (and `originalText.text` for the untranslated original) |
| `result.reviews[].relative_time_description` | `reviews[].relativePublishTime` |
| `result.reviews[].language` | `reviews[].text.languageCode` |
| `result.user_ratings_total` | `userRatingCount` |

Two differences matter for the research design rather than for the code.

**The new API exposes `originalText` alongside the translated `text`.** A replication can therefore record the untranslated original and the language actually detected, which would replace the locale-based translation heuristic used here with a verified measurement. This is the single most valuable improvement available to anyone repeating this protocol.

**Review counts remain capped and relevance-ranked.** The new API does not lift the five-review limit, so the sampling caveat carries over unchanged.

## 8. Known limitations of the collection design

To be reported in the manuscript rather than discovered by a reviewer.

1. **Five reviews per provider per stratum, relevance-ranked.** Not a random sample of each provider's reviews. Platform ranking favours longer and more engaged-with reviews, which interacts with the review-length diagnostics reported in the analysis.
2. **Stratum labels are query artefacts.** A provider is "creative" because a query in that list returned it, not because its business model was independently verified. Provider-level validation is required before the labels carry substantive interpretation.
3. **Unbalanced query lists.** Eighteen creative queries against thirteen mass-market queries; retrieval volume per stratum is not directly comparable.
4. **Radius-based sampling frame.** Providers outside the stated radius of a city centroid are unreachable, which under-samples peripheral and rural community-based tourism — a segment of substantive interest to the creative stratum.
5. **`DETAILS_LANG = "pt-BR"` triggers machine translation** for reviews written in other languages, and the returned `language` field records the delivered locale rather than the author's language.
6. **Non-reproducible sample.** Re-running yields a different corpus. The archived file is the object of analysis.
7. **Text Search ordering is undocumented.** Google does not publish the ranking function, so the composition of the provider frame cannot be fully characterised.